# 2. Daten analysieren

Gruppe 05 (SC02_Gruppe_05): Dominik Erbsland, Joel Liechti. ZHAW, CAS Information Engineering, Scripting.

Wir lesen die bereinigten Daten aus Notebook 1 wieder in ein **pandas DataFrame** und berechnen die Kennzahlen mit DataFrame-Methoden wie `value_counts`, `groupby` und `crosstab`. Jede Kennzahl speichern wir als eigene CSV-Datei in `data/processed/`. Notebook 3 stellt sie grafisch dar.

**Wichtig für die Interpretation:** Die Zahlen zeigen, wo und wann Schwarzfahrer *erwischt* wurden. Wo mehr kontrolliert wird, gibt es auch mehr Fälle. Die Anzahl Kontrollen ohne Befund kennen wir nicht.

In [ ]:
from pathlib import Path

import pandas as pd

PROCESSED = Path("../data/processed")

# Jahrgang wieder als ganze Zahl einlesen, sonst macht pandas daraus 2007.0
controls = pd.read_csv(PROCESSED / "controls_clean.csv", parse_dates=["datetime"], dtype={"birth_year": "Int64"})
print(controls.shape)
controls.head()

## 2.1 Überblick

In [ ]:
print("Anzahl Fälle:", len(controls))
print("Zeitraum:", controls["datetime"].min().date(), "bis", controls["datetime"].max().date())
print("Verkehrsunternehmen:", controls["company"].nunique())
print("Starthaltestellen:", controls["station_from"].nunique())

## 2.2 Geschlecht und Alter

In [ ]:
# Anteil in Prozent
gender = controls["gender"].value_counts()
print((gender / len(controls) * 100).round(1))
gender.to_csv(PROCESSED / "faelle_geschlecht.csv")

In [ ]:
# Anzahl Fälle pro Altersgruppe, aufgeteilt nach Geschlecht
age_gender = pd.crosstab(controls["age_group"], controls["gender"])

# Altersgruppen in der richtigen Reihenfolge statt alphabetisch
order = ["unter 18", "18 bis 25", "26 bis 35", "36 bis 50", "51 bis 65", "über 65"]
age_gender = age_gender.reindex(order)
age_gender.to_csv(PROCESSED / "faelle_alter_geschlecht.csv")
age_gender

In [ ]:
# Jahrgänge: unplausible Werte vor 1930 lassen wir weg
years = controls["birth_year"].value_counts().sort_index()
years = years[years.index >= 1930]
years.to_csv(PROCESSED / "faelle_jahrgang.csv")
print("Häufigste Jahrgänge:")
years.sort_values(ascending=False).head()

## 2.3 Wochentag und Tageszeit

In [ ]:
days = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]

# Tabelle: Zeilen = Wochentag, Spalten = Stunde, Werte = Anzahl Fälle
weekday_hour = pd.crosstab(controls["weekday"], controls["hour"])
weekday_hour.index = days
weekday_hour.to_csv(PROCESSED / "faelle_wochentag_stunde.csv")

print("Fälle pro Wochentag:")
print(weekday_hour.sum(axis=1))
print()
print("Stunde mit den meisten Fällen:", weekday_hour.sum().idxmax(), "Uhr")

In [ ]:
# Auffällig: Um 0 Uhr gibt es viele Fälle. Möglicherweise wurde bei fehlender Uhrzeit 00:00 eingetragen.
print("Fälle genau um 00:00:", (controls["datetime"].dt.strftime("%H:%M") == "00:00").sum())

## 2.4 Verkehrsunternehmen und Gründe

In [ ]:
companies = controls["company"].value_counts()
companies.to_csv(PROCESSED / "faelle_unternehmen.csv")
companies.head(10)

In [ ]:
reasons = controls["reason_cat"].value_counts()
reasons.to_csv(PROCESSED / "faelle_grund.csv")
reasons

In [ ]:
# Unterscheiden sich die Gründe je nach Unternehmen? Anteile in Prozent für die 5 grössten.
top5 = companies.head(5).index
top5_controls = controls[controls["company"].isin(top5)]
(pd.crosstab(top5_controls["company"], top5_controls["reason_cat"], normalize="index") * 100).round(1)

## 2.5 Haltestellen und Strecken

In [ ]:
# Fälle pro Starthaltestelle, mit Koordinaten für die Karte
stations = (controls.dropna(subset=["lat_from"])
            .groupby(["station_from", "lat_from", "lon_from"])
            .size()
            .reset_index(name="faelle")
            .sort_values("faelle", ascending=False))
stations.to_csv(PROCESSED / "faelle_haltestelle.csv", index=False)
stations.head(10)

In [ ]:
# Häufigste Strecken. Fälle mit gleichem Start und Ziel zählen wir nicht als Strecke.
trips = controls[controls["station_from"] != controls["station_to"]]
routes = (trips["station_from"] + " nach " + trips["station_to"]).value_counts()
routes.to_csv(PROCESSED / "faelle_strecke.csv")
routes.head(10)